In [ ]:
import pandas as pd
import numpy as np

In [ ]:
stories = pd.read_csv("RTN_Stories.csv")
stories

FileNotFoundError: [Errno 2] No such file or directory: 'RTN_Stories.csv'

In [ ]:
from openai import OpenAI
import pandas as pd
import json
from tqdm import tqdm
import os

# client = OpenAI(
#     api_key=
# )

In [ ]:
def extract_main_events(
    story_text: str,
    max_events: int = 6,
    model: str = "gpt-4.1-mini"
):
    """
    Extract main events from a personal narrative.
    Returns a list of event strings.
    """

    prompt = f"""
You are annotating personal narratives for a research study.

Definition of an event:
An event is a singular, concrete occurrence asserted to have happened at a particular time
(and often place) in the story. Events are typically expressed by verbs, but may also be
expressed by nouns or adjectives when they denote an occurrence. General habits, ongoing
states, hypothetical situations, and background facts are NOT events unless they function
as a specific story-like occurrence.

Task:
Extract the main events from the story below.

Guidelines:
- Extract only singular occurrences asserted to have happened.
- Do NOT extract general, habitual, or repeated actions.
- Do NOT extract background states or descriptions unless they clearly function as events.
- Negative events (things that failed or did not happen) MAY be included if they occur at a
  specific time and are narratively meaningful.
- Use simple, neutral language.
- Use past tense.
- Maintain chronological order.

Return ONLY a JSON array of strings.
Do not include explanations or extra text.

Story:
\"\"\"{story_text}\"\"\"
"""

    response = client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": "You extract structured events from narratives."},
            {"role": "user", "content": prompt}
        ]
    )

    content = response.choices[0].message.content.strip()

    try:
        events = json.loads(content)
        if isinstance(events, list):
            return events
        else:
            return []
    except json.JSONDecodeError:
        return []


In [ ]:
stories["events"] = None

for idx, row in tqdm(stories.iterrows(), total=len(stories)):
    events = extract_main_events(row["story_text"])
    stories.at[idx, "events"] = events

100%|██████████| 594/594 [14:15<00:00,  1.44s/it]


In [ ]:
stories

,story_id,story_text,events
0,0,"In college, I was figuring my life out. I didn...",[My guidance counselor told me I love to be in...
1,1,My name is I'm a professor at Harvard Business...,[The narrator taught finance at the business s...
2,2,from college at Brown University. And we were ...,"[He asked what the narrator wanted to study., ..."
3,3,"Before college, I was in a very traditional, l...","[I went to design school, I went to Carnegie M..."
4,4,two things I loved : new media and audio. And ...,[Tom Merit and the narrator decided to start a...
...,...,...,...
589,589,"After high school, I took a complete departure...","[I moved across the country, I went to the Coa..."
590,590,"The expectations on both sides, from my mother...",[I got to college and lost my support and scaf...
591,591,"help thinking , "" Well, one of those looks rea...",[My older brother was doing research on pediat...
592,592,acting. That means you're waiting tables or yo...,[He was a messenger boy delivering messages ar...


In [ ]:
stories.iloc[1]['events']

['I taught finance at the business school',
 'I taught taxation and tax policy at the law school',
 'I was on the faculty for about 19 years',
 'I came to love teaching',
 'I valued teaching more as I got older',
 'Teaching became a bigger part of my life',
 'I did my PhD',
 'I questioned myself during my PhD']

In [ ]:
stories.iloc[1]['story_text']

"My name is I'm a professor at Harvard Business School and Harvard Law School. I teach finance at the business school. And I teach taxation and tax policy at the law school. I've been on the faculty for about 19 years. So the really fun thing about teaching is it's a way to learn things. So when I teach something is when I understand the deeper. And so I just have really come to love it  and I think as I've gotten older, I value it even more and more. Teaching has become a bigger and bigger part of my life. I can't say there was like some grand design. There certainly wasn't. I'm not one of those people who saw my future with precision and was simply going down a path with clarity. That's just not the way I was. And in fact when I was doing my PhD was the worst. Because I had gone to business school. People I graduated with from business school are out there in the world making money and I was not. Like I was studying economics. And so that was when I questioned myself the most. And I 

In [ ]:
stories.to_csv("RTN_Stories.csv", index=False)

In [ ]:
stories["event_text"] = stories["events"].apply(
    lambda ev: " ".join(ev) if isinstance(ev, list) else ""
)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

model = SentenceTransformer("all-mpnet-base-v2")

event_embeddings = model.encode(
    stories["event_text"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

sim_event = cosine_similarity(event_embeddings)


full_embeddings = model.encode(
    stories["story_text"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

sim_full = cosine_similarity(full_embeddings)


full_sims = []
event_sims = []

n = len(stories)

for i in range(n):
    for j in range(i + 1, n):
        full_sims.append(sim_full[i, j])
        event_sims.append(sim_event[i, j])

full_low, full_high = np.percentile(full_sims, [33, 67])
event_low, event_high = np.percentile(event_sims, [33, 67])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [ ]:
from collections import defaultdict

bucketed_pairs = defaultdict(list)

for i in range(n):
    for j in range(i + 1, n):
        f = sim_full[i, j]
        e = sim_event[i, j]

        if f >= full_high and e >= event_high:
            bucket = "HH"   # high full, high event
        elif f >= full_high and e <= event_low:
            bucket = "HL"   # high full, low event
        elif f <= full_low and e >= event_high:
            bucket = "LH"   # low full, high event
        elif f <= full_low and e <= event_low:
            bucket = "LL"   # low full, low event
        else:
            continue

        bucketed_pairs[bucket].append((i, j))


bucket_rows = []

for bucket, pairs in bucketed_pairs.items():
    for i, j in pairs:
        bucket_rows.append({
            "story_i": i,
            "story_j": j,
            "bucket": bucket,
            "sim_full": sim_full[i, j],
            "sim_event": sim_event[i, j]
        })

buckets_df = pd.DataFrame(bucket_rows)


In [ ]:
buckets_df["bucket"].value_counts()


,count
bucket,
HH,29103
LL,25638
HL,13151
LH,10302


In [ ]:
import random
from collections import Counter

def sample_pairs_with_reuse_cap(bucketed_pairs, target_per_bucket, max_reuse=3, seed=13):
    """
    bucketed_pairs: dict bucket -> list[(i,j)]
    target_per_bucket: dict bucket -> int
    max_reuse: max number of times a story index can appear across ALL sampled pairs
    """
    rng = random.Random(seed)
    story_counts = Counter()
    sampled_all = []

    for bucket, k in target_per_bucket.items():
        candidates = bucketed_pairs[bucket].copy()
        rng.shuffle(candidates)

        chosen = []
        for i, j in candidates:
            if story_counts[i] < max_reuse and story_counts[j] < max_reuse:
                chosen.append((i, j))
                story_counts[i] += 1
                story_counts[j] += 1
                if len(chosen) == k:
                    break

        if len(chosen) < k:
            raise ValueError(
                f"Could not sample enough pairs for bucket {bucket}. "
                f"Got {len(chosen)} / {k}. Try increasing max_reuse or lowering k."
            )

        sampled_all.extend([(i, j, bucket) for (i, j) in chosen])

    return sampled_all


In [ ]:
TARGET_PER_BUCKET = {"HH": 100, "HL": 100, "LH": 100, "LL": 100}
sampled_pairs = sample_pairs_with_reuse_cap(bucketed_pairs, TARGET_PER_BUCKET, max_reuse=3, seed=13)

len(sampled_pairs)


400

In [ ]:
import pandas as pd

pool_df = pd.DataFrame([
    {
        "story_i": i,
        "story_j": j,
        "bucket": bucket,
        "story_i_id": stories.iloc[i].get("story_id", i),
        "story_j_id": stories.iloc[j].get("story_id", j),
        "sim_full": sim_full[i, j],
        "sim_event": sim_event[i, j],
    }
    for (i, j, bucket) in sampled_pairs
])

pool_df.to_csv("pair_pool_400_balanced.csv", index=False)

pool_df["bucket"].value_counts()


,count
bucket,
HH,100
HL,100
LH,100
LL,100


In [ ]:
pool_df

,story_i,story_j,bucket,story_i_id,story_j_id,sim_full,sim_event
0,0,55,HH,0,55,0.746363,0.483632
1,19,206,HH,19,206,0.489715,0.465746
2,88,489,HH,88,489,0.392211,0.362002
3,227,491,HH,227,491,0.434372,0.391325
4,28,301,HH,28,301,0.428111,0.350184
...,...,...,...,...,...,...,...
395,57,166,LL,57,166,0.224562,0.129754
396,101,479,LL,101,479,0.143250,0.049554
397,124,444,LL,124,444,0.264642,0.071468
398,441,577,LL,441,577,0.150036,0.064518


In [ ]:
unique_story_ids = pd.unique(
    pool_df[["story_i", "story_j"]].values.ravel()
)

num_unique_stories = len(unique_story_ids)

num_unique_stories


439

In [ ]:
pool_df = pool_df.copy()

pool_df["story_i_text"] = pool_df["story_i"].apply(
    lambda idx: stories.iloc[int(idx)]["story_text"]
)

pool_df["story_j_text"] = pool_df["story_j"].apply(
    lambda idx: stories.iloc[int(idx)]["story_text"]
)

pool_df = pool_df.drop(columns=["story_i_id", "story_j_id"])
pool_df.to_csv("pair_pool_400_balanced.csv", index=False)

pool_df

,story_i,story_j,bucket,sim_full,sim_event,story_i_text,story_j_text
0,0,55,HH,0.746363,0.483632,"In college, I was figuring my life out. I didn...",I knew during high school that I wanted to stu...
1,19,206,HH,0.489715,0.465746,with my guidance counselor and we decided that...,"playing country music , you know, with guys wi..."
2,88,489,HH,0.392211,0.362002,I went with what I thought I should do trying ...,exactly what I wanted to do. When I was going ...
3,227,491,HH,0.434372,0.391325,I ended up in technology by accident. I come f...,"Mine is the most straightforward, most unobsta..."
4,28,301,HH,0.428111,0.350184,"I feel like okay I'm aware of these issues , I...","We grew up kind of dysfunctional and poor, sti..."
...,...,...,...,...,...,...,...
395,57,166,LL,0.224562,0.129754,"July 20th, 1988, 27 years ago yesterday, I bro...",It was interesting because we were all really ...
396,101,479,LL,0.143250,0.049554,So I'm a highschool drop out. For real? Yeah. ...,And I can't screw things up any worse than the...
397,124,444,LL,0.264642,0.071468,A few years ago I had a bit of a scare. My app...,"I'm 53 years old , I'm trying to decide what I..."
398,441,577,LL,0.150036,0.064518,A lot of teachers feel like they have to be th...,"executive producer for a network. Um , that le..."


In [ ]:
pool_df.columns

Index(['story_i', 'story_j', 'bucket', 'sim_full', 'sim_event', 'story_i_text',
       'story_j_text'],
      dtype='object')

In [ ]:
# statistical analysis
# relation with each other between scores
# in a given pair, is there a correlation between sim_full and sim_event
# looks at Saldias et al (for number of participants)

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv("Final_Story_Pairs_400.csv")
df

,story_i,story_j,bucket,sim_full,sim_event,story_i_text,story_j_text
0,0,55,HH,0.746363,0.483632,"In college, I was figuring my life out. I didn...",I knew during high school that I wanted to stu...
1,19,206,HH,0.489715,0.465745,with my guidance counselor and we decided that...,"playing country music , you know, with guys wi..."
2,88,489,HH,0.392211,0.362002,I went with what I thought I should do trying ...,exactly what I wanted to do. When I was going ...
3,227,491,HH,0.434372,0.391325,I ended up in technology by accident. I come f...,"Mine is the most straightforward, most unobsta..."
4,28,301,HH,0.428111,0.350184,"I feel like okay I'm aware of these issues , I...","We grew up kind of dysfunctional and poor, sti..."
...,...,...,...,...,...,...,...
395,57,166,LL,0.224562,0.129754,"July 20th, 1988, 27 years ago yesterday, I bro...",It was interesting because we were all really ...
396,101,479,LL,0.143250,0.049554,So I'm a highschool drop out. For real? Yeah. ...,And I can't screw things up any worse than the...
397,124,444,LL,0.264642,0.071468,A few years ago I had a bit of a scare. My app...,"I'm 53 years old , I'm trying to decide what I..."
398,441,577,LL,0.150036,0.064518,A lot of teachers feel like they have to be th...,"executive producer for a network. Um , that le..."


In [ ]:
df.iloc[5]["story_i_text"]

"I wanted to be an artist when I was in high school , but I actually started in nursing school. We were really poor , and my dad died when I was six , and I saw my mom struggle working in a sewing factory. And I thought , well this is crazy to be a starving artist. I better do something practical. So I went to school for nursing. And photography and me was a better fit. It just felt like a better fit. So, started at the Miami Herald as a staff photographer, just from there moved on to do international stories, so I bounced around a lot. I traveled a whole lot. You know, when you can capture a moment, capture history , I don't know, just preserve that second of time, that's meaningful in some way. If it makes you feel, it's probably gon na make someone else feel too. But those were the moments that I always look for. And I feel like there's all different types of photography that you have to appreciate. You can do all these stylized beautiful perfectly composed pictures , but for me, jo

In [ ]:
df.iloc[5]["story_j_text"]

"I'm Mia Velasquez , I am co-founder and CEO of a company called Commit. I come from a non-entrepreneurial, almost not even business-focused background. I started school kind of in the mindset of I wan na be a doctor. And when you get to college , and I sat there and I'm like , okay, going into medicine is a huge commitment. And it's actually not really something that drives me. In school, I realized that what made me really happy and passionate and motivated me was working in teams to solve problems. And that kind of was really nebulous for me at the time of how you turn that into a career, until I found consulting. It didn't force me to choose one career , it allowed me to continue exploring, and trying different things. And I'm gon na be honest , I did not think I was ever going to be an entrepreneur. I left consulting to go to school, get my Masters. And Commit started while I was still on school. And then I graduated less than two months ago, and have been working on my startup fu

In [ ]:
import pandas as pd
import ast

# ----------------------------
# 1. Load CSVs
# ----------------------------
pairs_path = "Final_Story_Pairs_400.csv"
events_path = "RTN_Stories+events.csv"

pairs_df = pd.read_csv(pairs_path)
events_df = pd.read_csv(events_path)

# ----------------------------
# 2. Clean / prepare events
# ----------------------------
# Events are stored as stringified lists -> convert to list
def parse_events(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    return ast.literal_eval(x)

events_df["events"] = events_df["events"].apply(parse_events)

# Join events into a readable multi-line string (Qualtrics-friendly)
events_df["events_str"] = events_df["events"].apply(
    lambda evs: "\n".join(f"- {e}" for e in evs)
)

# Build lookup tables
story_text_map = dict(zip(events_df.story_id, events_df.story_text))
events_map = dict(zip(events_df.story_id, events_df.events_str))

# ----------------------------
# 3. Build Qualtrics-ready rows
# ----------------------------
rows = []

for idx, row in pairs_df.iterrows():
    story_i = row["story_i"]
    story_j = row["story_j"]

    rows.append({
        "PairID": f"P{idx:03d}",
        "FullA": story_text_map.get(story_i, ""),
        "FullB": story_text_map.get(story_j, ""),
        "EventsA": events_map.get(story_i, ""),
        "EventsB": events_map.get(story_j, "")
    })

qualtrics_df = pd.DataFrame(rows)

# ----------------------------
# 4. Save final CSV
# ----------------------------
output_path = "Qualtrics_Story_Pairs.csv"
qualtrics_df.to_csv(output_path, index=False)

print(f"Saved Qualtrics file to: {output_path}")
qualtrics_df.head()



Saved Qualtrics file to: Qualtrics_Story_Pairs.csv


,PairID,FullA,FullB,EventsA,EventsB
0,P000,"In college, I was figuring my life out. I didn...",I knew during high school that I wanted to stu...,- My guidance counselor told me I love to be i...,- I tried out the research field for a little ...
1,P001,with my guidance counselor and we decided that...,"playing country music , you know, with guys wi...",- I decided to become an architect with my gui...,- I started volunteering and working in zoos
2,P002,I went with what I thought I should do trying ...,exactly what I wanted to do. When I was going ...,- I started taking women's studies courses\n- ...,- Started rock climbing\n- Started doing bigge...
3,P003,I ended up in technology by accident. I come f...,"Mine is the most straightforward, most unobsta...",- I took courses at a junior college.\n- A par...,- completed an associates degree at Hawaii Com...
4,P004,"I feel like okay I'm aware of these issues , I...","We grew up kind of dysfunctional and poor, sti...",- I cried a lot\n- I meditated\n- I declared t...,- I ended up getting pregnant really young\n- ...


In [ ]:
# Qualtrics-ready merge script
# ----------------------------
# This cell:
# 1) Loads Final_Story_Pairs_400.csv and RTN_Stories+events.csv
# 2) Attaches full stories + event sequences to each pair
# 3) Preserves metadata (bucket, sim_full, sim_event)
# 4) Outputs a single CSV ready for Qualtrics Loop & Merge

import pandas as pd
import ast

# ----------------------------
# Paths (Colab)
# ----------------------------
PAIRS_PATH = "Final_Story_Pairs_400.csv"
EVENTS_PATH = "RTN_Stories+events.csv"
OUTPUT_PATH = "Qualtrics_Story_Pairs_FULL.csv"

# ----------------------------
# Load data
# ----------------------------
pairs_df = pd.read_csv(PAIRS_PATH)
events_df = pd.read_csv(EVENTS_PATH)

# ----------------------------
# Parse events (stored as stringified lists)
# ----------------------------
def parse_events(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    return ast.literal_eval(x)

events_df["events"] = events_df["events"].apply(parse_events)

# Join events into a readable multi-line string (Qualtrics-friendly)
events_df["events_str"] = events_df["events"].apply(
    lambda evs: "\n".join(f"- {e}" for e in evs)
)

# ----------------------------
# Build lookup tables
# ----------------------------
story_text_map = dict(zip(events_df["story_id"], events_df["story_text"]))
events_map = dict(zip(events_df["story_id"], events_df["events_str"]))

# ----------------------------
# Build final Qualtrics table
# ----------------------------
rows = []

for idx, row in pairs_df.iterrows():
    si, sj = row["story_i"], row["story_j"]

    rows.append({
        "PairID": f"P{idx:03d}",
        "story_i": si,
        "story_j": sj,
        "bucket": row["bucket"],
        "sim_full": row["sim_full"],
        "sim_event": row["sim_event"],
        "FullA": story_text_map.get(si, ""),
        "FullB": story_text_map.get(sj, ""),
        "EventsA": events_map.get(si, ""),
        "EventsB": events_map.get(sj, "")
    })

qualtrics_df = pd.DataFrame(rows)

# ----------------------------
# Save
# ----------------------------
qualtrics_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved Qualtrics CSV to: {OUTPUT_PATH}")
qualtrics_df.head()


Saved Qualtrics CSV to: Qualtrics_Story_Pairs_FULL.csv


,PairID,story_i,story_j,bucket,sim_full,sim_event,FullA,FullB,EventsA,EventsB
0,P000,0,55,HH,0.746363,0.483632,"In college, I was figuring my life out. I didn...",I knew during high school that I wanted to stu...,- My guidance counselor told me I love to be i...,- I tried out the research field for a little ...
1,P001,19,206,HH,0.489715,0.465745,with my guidance counselor and we decided that...,"playing country music , you know, with guys wi...",- I decided to become an architect with my gui...,- I started volunteering and working in zoos
2,P002,88,489,HH,0.392211,0.362002,I went with what I thought I should do trying ...,exactly what I wanted to do. When I was going ...,- I started taking women's studies courses\n- ...,- Started rock climbing\n- Started doing bigge...
3,P003,227,491,HH,0.434372,0.391325,I ended up in technology by accident. I come f...,"Mine is the most straightforward, most unobsta...",- I took courses at a junior college.\n- A par...,- completed an associates degree at Hawaii Com...
4,P004,28,301,HH,0.428111,0.350184,"I feel like okay I'm aware of these issues , I...","We grew up kind of dysfunctional and poor, sti...",- I cried a lot\n- I meditated\n- I declared t...,- I ended up getting pregnant really young\n- ...


In [ ]:
# --------------------------------------------------
# Sample up to 30 story PAIRS per bucket
# Goal: final CSV with <= 150 pairs total
# (e.g., HH, HL, LH, LL)
# --------------------------------------------------

import pandas as pd

INPUT_PATH = "Qualtrics_Story_Pairs_FULL.csv"
OUTPUT_PATH = "Qualtrics_Story_Pairs_sampled_30perBucket.csv"

# Load merged Qualtrics-ready file
df = pd.read_csv(INPUT_PATH)

# Sanity check
print("Pairs per bucket BEFORE sampling:")
print(df["bucket"].value_counts(), "\n")

# --------------------------------------------------
# Sampling: up to 30 per bucket
# --------------------------------------------------
SAMPLES_PER_BUCKET = 30
RANDOM_SEED = 42

sampled_dfs = []

for bucket, bucket_df in df.groupby("bucket"):
    n = min(SAMPLES_PER_BUCKET, len(bucket_df))
    sampled = bucket_df.sample(n=n, random_state=RANDOM_SEED)
    sampled_dfs.append(sampled)

final_df = pd.concat(sampled_dfs).reset_index(drop=True)

# --------------------------------------------------
# Shuffle final rows (important for Qualtrics)
# --------------------------------------------------
final_df = final_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# --------------------------------------------------
# Save
# --------------------------------------------------
final_df.to_csv(OUTPUT_PATH, index=False)

print("Pairs per bucket AFTER sampling:")
print(final_df["bucket"].value_counts())
print("\nTotal pairs:", len(final_df))
print(f"\nSaved sampled file to: {OUTPUT_PATH}")

final_df.head()


Pairs per bucket BEFORE sampling:
bucket
HH    100
HL    100
LH    100
LL    100
Name: count, dtype: int64 

Pairs per bucket AFTER sampling:
bucket
HL    30
HH    30
LH    30
LL    30
Name: count, dtype: int64

Total pairs: 120

Saved sampled file to: Qualtrics_Story_Pairs_sampled_30perBucket.csv


,PairID,story_i,story_j,bucket,sim_full,sim_event,FullA,FullB,EventsA,EventsB
0,P190,99,519,HL,0.461379,0.107206,I got a part-time job at the brewery. I was wo...,The point of education is to prepare kids for ...,- I got a part-time job at the brewery.\n- Our...,NaN
1,P177,106,460,HL,0.528713,0.048766,"whatever, is... is you 're learning. And yet, ...","I'm the founder of PixelsIO, which is a compan...",NaN,- I created technology while I was a grad stud...
2,P044,7,480,HH,0.421583,0.466924,We are creating soft multi-layered forms that ...,"So when I was your age, I think I had just dec...",- I had $300\n- I got a plane ticket to Bali f...,- I decided to try the field of biomedical eng...
3,P115,32,375,HL,0.464384,0.090594,"Welcome to Mastercard , I'm. I'm the executive...","inventing things. Just uh, when you live a lif...",NaN,- graduated from high school\n- lived out of m...
4,P040,344,431,HH,0.444965,0.304212,It's really easy to get caught up in believing...,"I figured , you know, the things that really m...","- I told authors don't make money, creative pe...",- I started a to-do app called Teuxdeux.\n- Te...


In [ ]:
final_df

,PairID,story_i,story_j,bucket,sim_full,sim_event,FullA,FullB,EventsA,EventsB
0,P190,99,519,HL,0.461379,0.107206,I got a part-time job at the brewery. I was wo...,The point of education is to prepare kids for ...,- I got a part-time job at the brewery.\n- Our...,NaN
1,P177,106,460,HL,0.528713,0.048766,"whatever, is... is you 're learning. And yet, ...","I'm the founder of PixelsIO, which is a compan...",NaN,- I created technology while I was a grad stud...
2,P044,7,480,HH,0.421583,0.466924,We are creating soft multi-layered forms that ...,"So when I was your age, I think I had just dec...",- I had $300\n- I got a plane ticket to Bali f...,- I decided to try the field of biomedical eng...
3,P115,32,375,HL,0.464384,0.090594,"Welcome to Mastercard , I'm. I'm the executive...","inventing things. Just uh, when you live a lif...",NaN,- graduated from high school\n- lived out of m...
4,P040,344,431,HH,0.444965,0.304212,It's really easy to get caught up in believing...,"I figured , you know, the things that really m...","- I told authors don't make money, creative pe...",- I started a to-do app called Teuxdeux.\n- Te...
...,...,...,...,...,...,...,...,...,...,...
115,P376,329,446,LL,0.245730,0.128984,"As a successful recording artist, how would yo...",I'm Brian. I farm full time for a living with ...,NaN,- Brian went into a retail job right out of co...
116,P090,79,209,HH,0.679818,0.587703,So did you always know that you were going to ...,"I have an education degree, and then math as a...",- ran an app store program\n- helped start a s...,- became a corporate trainer\n- ended up with ...
117,P370,110,121,LL,0.202158,0.103914,What can you say that is the most struggling w...,"What ended up happening my senior year , the V...",- She took on a ton of school loans to get thr...,- The Vice Principal greeted me at the door\n-...
118,P188,34,218,HL,0.603266,0.060005,So I grew up in the projects in Canarsie. Simi...,So I entered foster care when I was about thre...,NaN,- I entered foster care when I was about three...


In [ ]:
import pandas as pd

INPUT_PATH = "Qualtrics_Story_Pairs_FULL.csv"
OUTPUT_PATH = "Qualtrics_Story_Pairs_sampled_clean.csv"

SAMPLES_PER_BUCKET = 30
RANDOM_SEED = 42

# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(INPUT_PATH)

# ----------------------------
# 1. Drop pairs with missing events
# ----------------------------
clean_df = df.dropna(subset=["EventsA", "EventsB"]).copy()

print("After dropping NaN events:")
print(clean_df["bucket"].value_counts(), "\n")

# ----------------------------
# 2. Sample up to 30 per bucket
# ----------------------------
sampled = []

for bucket, bucket_df in clean_df.groupby("bucket"):
    n = min(SAMPLES_PER_BUCKET, len(bucket_df))
    sampled.append(bucket_df.sample(n=n, random_state=RANDOM_SEED))

final_df = pd.concat(sampled).reset_index(drop=True)

# ----------------------------
# 3. Shuffle rows (important for Qualtrics)
# ----------------------------
final_df = final_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# ----------------------------
# 4. Sanity checks
# ----------------------------
print("Final pairs per bucket:")
print(final_df["bucket"].value_counts())
print("\nTotal pairs:", len(final_df))

assert final_df["EventsA"].isna().sum() == 0
assert final_df["EventsB"].isna().sum() == 0

# ----------------------------
# 5. Save
# ----------------------------
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved CLEAN Qualtrics file to:\n{OUTPUT_PATH}")

final_df.head()


After dropping NaN events:
bucket
HH    92
LH    88
LL    57
HL    37
Name: count, dtype: int64 

Final pairs per bucket:
bucket
HL    30
HH    30
LH    30
LL    30
Name: count, dtype: int64

Total pairs: 120

Saved CLEAN Qualtrics file to:
Qualtrics_Story_Pairs_sampled_clean.csv


,PairID,story_i,story_j,bucket,sim_full,sim_event,FullA,FullB,EventsA,EventsB
0,P131,246,455,HL,0.407780,0.123518,"and uh, loved it. You know, I was in one of th...",""" to be a flight attendant. '' So I was applyi...",- I was in one of those graduate programs\n- I...,- I was applying for the flight attendant posi...
1,P189,4,242,HL,0.446355,0.069382,two things I loved : new media and audio. And ...,I took a super non-traditional path to get her...,- Tom Merit and the narrator decided to start ...,- I graduated with a degree in physiological s...
2,P000,0,55,HH,0.746363,0.483632,"In college, I was figuring my life out. I didn...",I knew during high school that I wanted to stu...,- My guidance counselor told me I love to be i...,- I tried out the research field for a little ...
3,P196,222,469,HL,0.454492,0.125337,I'm from Virginia originally. That's where my ...,"Well, my name is Julian Serrano. I originally ...",- I moved to New York after graduation\n- I au...,- Julian Serrano went to culinary school when ...
4,P072,18,482,HH,0.381345,0.320885,Well... what's the chances of makin ' it? Abou...,"I went to school for welding, design, pouring ...",- He said to go to school and get all the degr...,"- I went to school for welding, design, pourin..."


In [ ]:
final_df.iloc[17]["FullA"]

"We are creating soft multi-layered forms that interact with environments. And create a space in cities that open up our imagination and allow us to think about a different way of being. So I had $ 300 and for graduation I got a plane ticket to Bali. And I went off on my own and started walking through the rice fields until I found a little house to rent. I went as a painter, I shipped all my paints and they got lost , and that is how I became a sculpture, because I had no paint. I was in a fishing village , they had fishing twine, and began just experimenting, and from there I had this dream of bringing it back home. I applied to seven art schools and I was rejected by all seven. And that turned out to be the luckiest thing ever in my life, because it forced me to have to learn how to teach myself. The education to do what I do, there is no program. If I had gone to a program, it might have led me somewhere , but it wouldn't have been as authentic a direction. Any kind of life where y

In [ ]:
final_df.iloc[17]["EventsA"]

'- I had $300\n- I got a plane ticket to Bali for graduation\n- I went off on my own\n- I started walking through the rice fields\n- I found a little house to rent\n- I shipped all my paints\n- My paints got lost\n- I became a sculpture because I had no paint\n- I began experimenting with fishing twine\n- I had a dream of bringing it back home\n- I applied to seven art schools\n- I was rejected by all seven art schools\n- I had to learn how to teach myself\n- I had to teach myself how to hear my own voice'

In [ ]:
import pandas as pd

# === CHANGE THIS PATH TO YOUR FILE ===
INPUT_PATH = "Qualtrics_Story_Pairs_sampled_clean.csv"
OUTPUT_PATH = "Qualtrics_Story_Pairs_sampled_clean.csv"

df = pd.read_csv(INPUT_PATH)

def clean_events(text):
    if pd.isna(text):
        return ""
    text = str(text)

    # Handle escaped and real newlines
    text = text.replace("\\n", "\n")

    # Split on newlines, remove bullets/dashes, trim whitespace
    parts = [
        p.strip(" -•\t ")
        for p in text.split("\n")
        if p.strip()
    ]

    # Join into a single readable line
    return " — ".join(parts)

df["EventsA"] = df["EventsA"].apply(clean_events)
df["EventsB"] = df["EventsB"].apply(clean_events)

df.to_csv(OUTPUT_PATH, index=False)

print("Saved cleaned file to:", OUTPUT_PATH)
print(df[["EventsA", "EventsB"]].head())


Saved cleaned file to: Qualtrics_Story_Pairs_sampled_clean.csv
                                             EventsA  \
0  I was in one of those graduate programs — I wa...   
1  Tom Merit and the narrator decided to start a ...   
2  My guidance counselor told me I love to be in ...   
3  I moved to New York after graduation — I audit...   
4  He said to go to school and get all the degree...   

                                             EventsB  
0  I was applying for the flight attendant positi...  
1  I graduated with a degree in physiological sci...  
2  I tried out the research field for a little wh...  
3  Julian Serrano went to culinary school when he...  
4  I went to school for welding, design, pouring ...  


In [ ]:
import pandas as pd

df = pd.read_csv("Qualtrics_Story_Pairs_sampled_clean.csv")

df.iloc[10]['EventsB']

'Developed a kit called LilyPad Arduino that lets you sew electronics into your clothes — Started to play and build stuff with electronics and textiles — Felt a tremendous a-ha moment connecting crafty hobbies with electronics and computer science'

In [ ]:
print('-')

-


In [ ]:
import re

SEP = re.compile(r"""
    (?:\s+[-–—]\s+)      # dash-like separator surrounded by whitespace
  | (?:^\s*[-–—]\s+)     # leading bullet dash at start
  | (?:\n\s*[-–—]\s+)    # leading bullet dash after newline
""", re.VERBOSE)

def has_at_least_two_events(s):
    if not isinstance(s, str) or not s.strip():
        return False
    parts = [p.strip() for p in SEP.split(s.strip()) if p.strip()]
    return len(parts) >= 2

df = df[
    df["EventsA"].apply(has_at_least_two_events) &
    df["EventsB"].apply(has_at_least_two_events)
].reset_index(drop=True)

df


,PairID,story_i,story_j,bucket,sim_full,sim_event,FullA,FullB,EventsA,EventsB
0,P131,246,455,HL,0.407780,0.123518,"and uh, loved it. You know, I was in one of th...",""" to be a flight attendant. '' So I was applyi...",I was in one of those graduate programs — I wa...,I was applying for the flight attendant positi...
1,P000,0,55,HH,0.746363,0.483632,"In college, I was figuring my life out. I didn...",I knew during high school that I wanted to stu...,My guidance counselor told me I love to be in ...,I tried out the research field for a little wh...
2,P072,18,482,HH,0.381345,0.320885,Well... what's the chances of makin ' it? Abou...,"I went to school for welding, design, pouring ...",He said to go to school and get all the degree...,"I went to school for welding, design, pouring ..."
3,P212,8,523,LH,0.141156,0.423245,"you know, heavy civil tunneling for-for railro...",much like you -- went through uni and I was fa...,I was going to school at Colorado State Univer...,went tree-planting — spent up to two months li...
4,P292,373,385,LH,0.218787,0.400481,"I was already a pretty avid diver , and enjoye...",So going back into the time when you were firs...,came down to Miami — went to law school — prac...,I came out and had a film — I started looking ...
...,...,...,...,...,...,...,...,...,...,...
93,P315,444,459,LL,0.225717,0.135852,"I'm 53 years old , I'm trying to decide what I...",the Olympics. So I always wanted to be on an O...,Grandfather asked if they wanted to plant a ga...,I went on the Sunday morning bike ride — The o...
94,P013,0,59,HH,0.390628,0.427071,"In college, I was figuring my life out. I didn...",I worked in technology up until 1998 in which ...,My guidance counselor told me I love to be in ...,I struck out on my own — I built an engineerin...
95,P361,5,580,LL,0.253082,0.104825,So at Alchemy we are taking the concept of che...,where you weren't truly able to talk about emo...,went and worked at Detroit Country Day School ...,worked up the courage to read a poem at an ope...
96,P182,87,168,HL,0.432051,0.086877,So I'm a program manager at the Chan Zuckerber...,"My name is Tom Arviso, Jr. , I'm the CEO of th...",I spent 12 years in state corrections — I came...,"Tom Arviso, Jr. started as a college intern at..."


In [ ]:
df.to_csv("RTN_SURVEY_FINAL.csv", index=False)